In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ROOT = Path("KM_output_dstrainPerStep1e-3_strainrate60")
PLOT_DIR = ROOT / "KM_plots"

# Constants for stored-energy and critical-radius proxy
G_SHEAR = 32.7e9       # Pa
BURGERS = 2.96e-10     # m
GB_ENERGY = 0.5        # J/m^2


def parse_number_from_text(text):
    """
    Extract numbers from folder names such as:
        rho_1e16
        rho_init_1e16
        T_1073
        gdot_0p1
    """

    text = str(text).replace("p", ".").replace(",", ".")

    match = re.search(r"[-+]?\d*\.?\d+(?:e[-+]?\d+)?", text)

    if match is None:
        return np.nan

    return float(match.group(0))


def parse_metadata_from_path(csv_path):
    rho_init = np.nan
    T = np.nan
    strain_rate = np.nan

    for part in csv_path.parts:
        lower = part.lower()

        if lower.startswith("rho_") or lower.startswith("rhoinit_") or lower.startswith("rho_init_"):
            rho_init = parse_number_from_text(part)

        elif lower.startswith("t_") or lower.startswith("temp_"):
            T = parse_number_from_text(part)

        elif lower.startswith("gdot_") or lower.startswith("strain_rate_") or lower.startswith("rate_"):
            strain_rate = parse_number_from_text(part)

    return rho_init, T, strain_rate


def first_positive_value(series):
    values = pd.to_numeric(series, errors="coerce")
    values = values[np.isfinite(values) & (values > 0)]

    if values.empty:
        return np.nan

    return values.iloc[0]


def safe_tag(value):
    return f"{value:g}".replace(".", "p").replace("-", "m")


def load_km_results(root):
    csv_files = sorted(root.rglob("km.csv"))

    if not csv_files:
        raise FileNotFoundError(f"No km.csv files found under: {root}")

    all_data = []

    for csv_path in csv_files:
        df = pd.read_csv(csv_path)

        required = ["time", "T_avg", "gdot_avg", "rho_avg", "tau_avg"]

        missing = [col for col in required if col not in df.columns]
        if missing:
            print(f"Skipping {csv_path}: missing {missing}")
            continue

        rho_init_from_path, T_from_path, rate_from_path = parse_metadata_from_path(csv_path)

        out = pd.DataFrame()
        out["time"] = pd.to_numeric(df["time"], errors="coerce")
        out["T"] = pd.to_numeric(df["T_avg"], errors="coerce")
        out["strain_rate"] = pd.to_numeric(df["gdot_avg"], errors="coerce")
        out["rho_avg"] = pd.to_numeric(df["rho_avg"], errors="coerce")
        out["tau_avg"] = pd.to_numeric(df["tau_avg"], errors="coerce")

        if "k1_avg" in df.columns:
            out["k1_avg"] = pd.to_numeric(df["k1_avg"], errors="coerce")
        else:
            out["k1_avg"] = np.nan

        if "k2dyn_t_avg" in df.columns:
            out["k2dyn_t_avg"] = pd.to_numeric(df["k2dyn_t_avg"], errors="coerce")
        else:
            out["k2dyn_t_avg"] = np.nan

        if np.isfinite(T_from_path):
            out["T"] = T_from_path

        if np.isfinite(rate_from_path):
            out["strain_rate"] = rate_from_path

        if np.isfinite(rho_init_from_path):
            out["rho_init"] = rho_init_from_path
        else:
            out["rho_init"] = np.nan

        out["source_file"] = str(csv_path)

        # Remove the initial zero row from CSV output
        out = out.dropna(subset=["time", "T", "strain_rate", "rho_avg"])
        out = out[out["rho_avg"] > 0].copy()

        if out.empty:
            print(f"Skipping {csv_path}: no positive rho_avg values")
            continue

        all_data.append(out)

    if not all_data:
        raise RuntimeError("No usable KM data found.")

    data = pd.concat(all_data, ignore_index=True)

    data = data.sort_values(
        ["rho_init", "T", "strain_rate", "time"],
        na_position="last",
    ).reset_index(drop=True)

    group_cols = ["rho_init", "T", "strain_rate"]

    data["rho0"] = data.groupby(group_cols, dropna=False)["rho_avg"].transform(
        first_positive_value
    )

    data["tau0"] = data.groupby(group_cols, dropna=False)["tau_avg"].transform(
        first_positive_value
    )

    data["rho_norm"] = data["rho_avg"] / data["rho0"]
    data["tau_norm"] = data["tau_avg"] / data["tau0"]

    data["accumulated_strain"] = data["strain_rate"] * data["time"]

    # Stored-energy proxy from KM rho
    data["stored_energy"] = 0.5 * G_SHEAR * BURGERS**2 * data["rho_avg"]

    # Critical-radius proxy:
    # r_crit = sigma / stored_energy
    data["rcrit_m"] = GB_ENERGY / data["stored_energy"]
    data["rcrit_nm"] = data["rcrit_m"] * 1.0e9

    return data


def plot_fixed_temperature(data, x_col, y_col, y_label, out_dir, log_y=False):
    plot_dir = out_dir / f"{y_col}_vs_{x_col}_fixed_T"
    plot_dir.mkdir(parents=True, exist_ok=True)

    for rho_init, group_rho in data.groupby("rho_init", dropna=False):
        for T, group_T in group_rho.groupby("T"):
            plt.figure()

            for rate, curve in group_T.groupby("strain_rate"):
                curve = curve.sort_values(x_col)
                plt.plot(curve[x_col], curve[y_col], label=f"{rate:g} 1/s")

            if x_col == "time":
                plt.xlabel("time, s")
            elif x_col == "accumulated_strain":
                plt.xlabel("accumulated strain")
            else:
                plt.xlabel(x_col)

            plt.ylabel(y_label)

            if np.isfinite(rho_init):
                title = f"{y_label}, T = {T:g} K, rho_init = {rho_init:.1e}"
            else:
                title = f"{y_label}, T = {T:g} K"

            plt.title(title)
            plt.legend(title="strain rate")

            if log_y:
                plt.yscale("log")

            plt.tight_layout()

            rho_tag = "rho_unknown" if not np.isfinite(rho_init) else f"rho_{rho_init:.1e}"
            out_path = plot_dir / f"{rho_tag}_T_{safe_tag(T)}_{y_col}_vs_{x_col}.png"
            plt.savefig(out_path, dpi=300)
            plt.close()


def plot_fixed_strain_rate(data, x_col, y_col, y_label, out_dir, log_y=False):
    plot_dir = out_dir / f"{y_col}_vs_{x_col}_fixed_gdot"
    plot_dir.mkdir(parents=True, exist_ok=True)

    for rho_init, group_rho in data.groupby("rho_init", dropna=False):
        for rate, group_rate in group_rho.groupby("strain_rate"):
            plt.figure()

            for T, curve in group_rate.groupby("T"):
                curve = curve.sort_values(x_col)
                plt.plot(curve[x_col], curve[y_col], label=f"{T:g} K")

            if x_col == "time":
                plt.xlabel("time, s")
            elif x_col == "accumulated_strain":
                plt.xlabel("accumulated strain")
            else:
                plt.xlabel(x_col)

            plt.ylabel(y_label)

            if np.isfinite(rho_init):
                title = f"{y_label}, strain rate = {rate:g} 1/s, rho_init = {rho_init:.1e}"
            else:
                title = f"{y_label}, strain rate = {rate:g} 1/s"

            plt.title(title)
            plt.legend(title="temperature")

            if log_y:
                plt.yscale("log")

            plt.tight_layout()

            rho_tag = "rho_unknown" if not np.isfinite(rho_init) else f"rho_{rho_init:.1e}"
            out_path = plot_dir / f"{rho_tag}_gdot_{safe_tag(rate)}_{y_col}_vs_{x_col}.png"
            plt.savefig(out_path, dpi=300)
            plt.close()


def plot_final_heatmap(data, y_col, y_label, out_dir):
    plot_dir = out_dir / f"final_{y_col}_heatmap"
    plot_dir.mkdir(parents=True, exist_ok=True)

    final_rows = (
        data.sort_values("time")
        .groupby(["rho_init", "T", "strain_rate"], as_index=False, dropna=False)
        .tail(1)
    )

    for rho_init, group in final_rows.groupby("rho_init", dropna=False):
        pivot = group.pivot_table(
            index="T",
            columns="strain_rate",
            values=y_col,
            aggfunc="mean",
        )

        pivot = pivot.sort_index().sort_index(axis=1)

        plt.figure()
        image = plt.imshow(
            pivot.values,
            origin="lower",
            aspect="auto",
        )

        plt.colorbar(image, label=y_label)

        plt.xticks(
            ticks=np.arange(len(pivot.columns)),
            labels=[f"{x:g}" for x in pivot.columns],
            rotation=45,
        )

        plt.yticks(
            ticks=np.arange(len(pivot.index)),
            labels=[f"{x:g}" for x in pivot.index],
        )

        plt.xlabel("strain rate, 1/s")
        plt.ylabel("temperature, K")

        if np.isfinite(rho_init):
            plt.title(f"Final {y_label}, rho_init = {rho_init:.1e}")
            rho_tag = f"rho_{rho_init:.1e}"
        else:
            plt.title(f"Final {y_label}")
            rho_tag = "rho_unknown"

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                value = pivot.values[i, j]
                if np.isfinite(value):
                    plt.text(j, i, f"{value:.2g}", ha="center", va="center")

        plt.tight_layout()
        plt.savefig(plot_dir / f"{rho_tag}_final_{y_col}.png", dpi=300)
        plt.close()


def main():
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    data = load_km_results(ROOT)

    combined_path = PLOT_DIR / "combined_km_results.csv"
    data.to_csv(combined_path, index=False)

    final_summary = (
        data.sort_values("time")
        .groupby(["rho_init", "T", "strain_rate"], as_index=False, dropna=False)
        .tail(1)
        .sort_values(["rho_init", "T", "strain_rate"], na_position="last")
    )

    final_summary_path = PLOT_DIR / "final_km_summary.csv"
    final_summary.to_csv(final_summary_path, index=False)

    # rho plots
    for x_col in ["time", "accumulated_strain"]:
        plot_fixed_temperature(
            data,
            x_col=x_col,
            y_col="rho_avg",
            y_label=r"$\rho_{avg}$, 1/m$^2$",
            out_dir=PLOT_DIR,
            log_y=True,
        )

        plot_fixed_strain_rate(
            data,
            x_col=x_col,
            y_col="rho_avg",
            y_label=r"$\rho_{avg}$, 1/m$^2$",
            out_dir=PLOT_DIR,
            log_y=True,
        )

        plot_fixed_temperature(
            data,
            x_col=x_col,
            y_col="rho_norm",
            y_label=r"$\rho / \rho_0$",
            out_dir=PLOT_DIR,
            log_y=False,
        )

        plot_fixed_strain_rate(
            data,
            x_col=x_col,
            y_col="rho_norm",
            y_label=r"$\rho / \rho_0$",
            out_dir=PLOT_DIR,
            log_y=False,
        )

    # tau plots
    plot_fixed_temperature(
        data,
        x_col="time",
        y_col="tau_avg",
        y_label=r"$\tau_{avg}$, Pa",
        out_dir=PLOT_DIR,
        log_y=False,
    )

    plot_fixed_strain_rate(
        data,
        x_col="time",
        y_col="tau_avg",
        y_label=r"$\tau_{avg}$, Pa",
        out_dir=PLOT_DIR,
        log_y=False,
    )

    # stored-energy / critical-radius proxy plots
    plot_fixed_temperature(
        data,
        x_col="time",
        y_col="stored_energy",
        y_label=r"stored-energy proxy, J/m$^3$",
        out_dir=PLOT_DIR,
        log_y=True,
    )

    plot_fixed_strain_rate(
        data,
        x_col="time",
        y_col="stored_energy",
        y_label=r"stored-energy proxy, J/m$^3$",
        out_dir=PLOT_DIR,
        log_y=True,
    )

    plot_fixed_temperature(
        data,
        x_col="time",
        y_col="rcrit_nm",
        y_label="critical-radius proxy, nm",
        out_dir=PLOT_DIR,
        log_y=True,
    )

    plot_fixed_strain_rate(
        data,
        x_col="time",
        y_col="rcrit_nm",
        y_label="critical-radius proxy, nm",
        out_dir=PLOT_DIR,
        log_y=True,
    )

    # heatmaps
    plot_final_heatmap(
        data,
        y_col="rho_avg",
        y_label=r"$\rho_{avg}$, 1/m$^2$",
        out_dir=PLOT_DIR,
    )

    plot_final_heatmap(
        data,
        y_col="rho_norm",
        y_label=r"$\rho / \rho_0$",
        out_dir=PLOT_DIR,
    )

    plot_final_heatmap(
        data,
        y_col="stored_energy",
        y_label=r"stored-energy proxy, J/m$^3$",
        out_dir=PLOT_DIR,
    )

    plot_final_heatmap(
        data,
        y_col="rcrit_nm",
        y_label="critical-radius proxy, nm",
        out_dir=PLOT_DIR,
    )

    print("Finished KM plotting.")
    print()
    print(f"Combined data saved to:")
    print(f"  {combined_path}")
    print()
    print(f"Final summary saved to:")
    print(f"  {final_summary_path}")
    print()
    print(f"Plots saved under:")
    print(f"  {PLOT_DIR}")


if __name__ == "__main__":
    main()